# Proyecto Sprint 10: Modelo de Clasificación para Planes Megaline

## Descripción general
La compañía móvil Megaline busca incentivar a sus usuarios con planes antiguos a cambiarse a sus nuevos planes: **Smart** o **Ultra**. 

En este proyecto cuento con el registro mensual del comportamiento de clientes que ya usan estos planes modernos. Dado que la variable objetivo a predecir es categórica binaria (`is_ultra`: 0 para Smart y 1 para Ultra), se trata de un problema de **aprendizaje supervisado de clasificación**.

### Objetivo
Construir un modelo de clasificación capaz de analizar las llamadas, minutos, mensajes y datos consumidos para recomendar el plan adecuado, superando el umbral mínimo de exactitud (*accuracy*) de **0.75** en el conjunto de prueba.

### Pasos a seguir:
1. Cargar y examinar los datos de `/datasets/users_behavior.csv`.
2. Segmentar los datos en tres conjuntos: entrenamiento (60%), validación (20%) y prueba (20%).
3. Entrenar y ajustar hiperparámetros en tres modelos distintos:
   - Árbol de decisión
   - Bosque aleatorio
   - Regresión logística
4. Evaluar el modelo seleccionado en el conjunto de prueba.
5. Realizar una prueba de cordura (*sanity check*) frente a un modelo base.

In [14]:
# Importamos todas las librerías que usaremos
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier

## Paso 1: Carga y exploración de los datos
Procedo a cargar el dataset proporcionado por Megaline para comprobar su estructura, dimensiones y confirmar que no contiene datos nulos o tipos de datos inconsistentes.

In [15]:
# Cargamos el archivo CSV usando una estructura try-except para que funcione tanto en el servidor de la plataforma como en local
try:
    df = pd.read_csv('/datasets/users_behavior.csv')
except:
    df = pd.read_csv('users_behavior.csv')

# Revisamos las primeras filas del dataset para verificar que las columnas cargaron bien
print("Primeras filas del conjunto de datos:")
display(df.head())

# Verificamos la cantidad de filas, columnas y tipos de datos
print("\nEstructura general del dataset:")
df.info()

# Revisamos la distribución de clases en la variable objetivo 'is_ultra'
print("\nDistribución porcentual de los planes:")
print(df['is_ultra'].value_counts(normalize=True))

Primeras filas del conjunto de datos:


,calls,minutes,messages,mb_used,is_ultra
0,40.0,311.90,83.0,19915.42,0
1,85.0,516.75,56.0,22696.96,0
2,77.0,467.66,86.0,21060.45,0
3,106.0,745.53,81.0,8437.39,1
4,66.0,418.74,1.0,14502.75,0



Estructura general del dataset:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3214 entries, 0 to 3213
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   calls     3214 non-null   float64
 1   minutes   3214 non-null   float64
 2   messages  3214 non-null   float64
 3   mb_used   3214 non-null   float64
 4   is_ultra  3214 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 125.7 KB

Distribución porcentual de los planes:
0    0.693528
1    0.306472
Name: is_ultra, dtype: float64


### Observaciones iniciales:
- El dataset contiene **3,214 filas** y **5 columnas**.
- No hay valores ausentes (`Non-Null Count` es 3214 en todas las columnas), lo cual concuerda con las instrucciones que indican que los datos ya fueron preprocesados.
- Las variables predictoras (`features`) son: `calls`, `minutes`, `messages` y `mb_used`.
- La variable objetivo (`target`) es `is_ultra`, donde:
  - `0` (Smart) representa aproximadamente el **69.35%** de los datos.
  - `1` (Ultra) representa el **30.65%** restante.
- Hay un desbalance de clases moderado (casi 70/30), lo cual será muy importante tener en cuenta durante la prueba de cordura.

## Paso 2: Segmentación de los datos
Dado que no tenemos un conjunto de prueba provisto por separado, debemos dividir el dataset original en tres partes:
- **Entrenamiento (60%)**: para ajustar los parámetros internos de los modelos.
- **Validación (20%)**: para comparar algoritmos y afinar los hiperparámetros.
- **Prueba (20%)**: reservado exclusivamente para la evaluación final no sesgada.

Para lograr esta proporción 3:1:1 (60% - 20% - 20%), aplicaremos `train_test_split` en dos pasos sucesivos con `random_state=12345` para asegurar que los resultados sean reproducibles.

In [16]:
# Separamos las características (features) quitando la columna objetivo 'is_ultra'
features = df.drop(['is_ultra'], axis=1)

# Guardamos únicamente la columna objetivo en target
target = df['is_ultra']

# Primer split: 60% para entrenamiento y 40% temporal (que luego contendrá validación y prueba)
features_train, features_temp, target_train, target_temp = train_test_split(
    features, target, test_size=0.40, random_state=12345
)

# Segundo split: dividimos el 40% temporal en dos partes iguales (50% cada una -> 20% del total)
features_valid, features_test, target_valid, target_test = train_test_split(
    features_temp, target_temp, test_size=0.50, random_state=12345
)

# Imprimimos las dimensiones resultantes para asegurar que la división cumple con el 60/20/20
print(f"Dimensiones entrenamiento : {features_train.shape} ({len(features_train)/len(df):.0%})")
print(f"Dimensiones validación    : {features_valid.shape} ({len(features_valid)/len(df):.0%})")
print(f"Dimensiones prueba        : {features_test.shape} ({len(features_test)/len(df):.0%})")

Dimensiones entrenamiento : (1928, 4) (60%)
Dimensiones validación    : (643, 4) (20%)
Dimensiones prueba        : (643, 4) (20%)


## Paso 3: Investigación y ajuste de modelos
Ahora voy a entrenar tres algoritmos de clasificación distintos y a medir su desempeño sobre el conjunto de validación (`features_valid`, `target_valid`) ajustando sus hiperparámetros:
1. **Árbol de Decisión (`DecisionTreeClassifier`)**
2. **Bosque Aleatorio (`RandomForestClassifier`)**
3. **Regresión Logística (`LogisticRegression`)**

### Árbol de Decisión
Ajustamos el hiperparámetro `max_depth` (profundidad máxima) desde 1 hasta 10 para identificar el punto óptimo donde el modelo aprende bien sin caer en sobreajuste (*overfitting*).

In [17]:
# Inicializamos variables para almacenar la mejor configuración del árbol
best_tree_model = None
best_tree_depth = 0
best_tree_accuracy = 0.0

# Iteramos sobre profundidades de 1 a 10
for depth in range(1, 11):
    # Creamos el modelo con la profundidad actual y fijamos la semilla
    tree = DecisionTreeClassifier(max_depth=depth, random_state=12345)
    
    # Entrenamos el modelo con los datos de entrenamiento
    tree.fit(features_train, target_train)
    
    # Obtenemos predicciones en el conjunto de validación
    predictions_valid = tree.predict(features_valid)
    
    # Calculamos la exactitud
    acc = accuracy_score(target_valid, predictions_valid)
    
    # Si esta profundidad supera el mejor accuracy previo, guardamos el modelo y su valor
    if acc > best_tree_accuracy:
        best_tree_model = tree
        best_tree_depth = depth
        best_tree_accuracy = acc

print(f"Mejor Árbol de Decisión:")
print(f"Profundidad óptima (max_depth): {best_tree_depth}")
print(f"Exactitud en validación: {best_tree_accuracy:.4f}")

Mejor Árbol de Decisión:
Profundidad óptima (max_depth): 3
Exactitud en validación: 0.7854


### Bosque Aleatorio
Un bosque aleatorio suele dar mayor estabilidad al promediar múltiples árboles. Evaluaremos combinaciones de número de árboles (`n_estimators` de 10 a 50) y profundidad (`max_depth` de 1 a 10).

In [19]:
# Inicializamos variables para guardar el mejor bosque aleatorio
best_rf_model = None
best_rf_est = 0
best_rf_depth = 0
best_rf_accuracy = 0.0

# Bucle anidado para explorar estimadores y profundidad
for est in range(10, 51, 10):
    for depth in range(1, 11):
        # Configuramos el bosque aleatorio con los hiperparámetros actuales
        rf = RandomForestClassifier(n_estimators=est, max_depth=depth, random_state=12345)
        
        # Entrenamos el modelo
        rf.fit(features_train, target_train)
        
        # Predecimos sobre el conjunto de validación
        predictions_valid = rf.predict(features_valid)
        
        # Calculamos la exactitud
        acc = accuracy_score(target_valid, predictions_valid)
        
        # Si obtenemos un mejor accuracy, actualizamos el mejor modelo
        if acc > best_rf_accuracy:
            best_rf_model = rf
            best_rf_est = est
            best_rf_depth = depth
            best_rf_accuracy = acc

print(f"Mejor Bosque Aleatorio:")
print(f"Estimadores (n_estimators): {best_rf_est}")
print(f"Profundidad (max_depth): {best_rf_depth}")
print(f"Exactitud en validación: {best_rf_accuracy:.4f}")

Mejor Bosque Aleatorio:
Estimadores (n_estimators): 40
Profundidad (max_depth): 8
Exactitud en validación: 0.8087


### Regresión Logística
Probamos la regresión logística para ver si un algoritmo lineal simple es suficiente para separar ambas clases.

In [20]:
# Inicializamos la regresión logística usando solver='liblinear' que funciona bien para conjuntos medianos
log_model = LogisticRegression(solver='liblinear', random_state=12345)

# Entrenamos con el conjunto de entrenamiento
log_model.fit(features_train, target_train)

# Predecimos en validación
predictions_log = log_model.predict(features_valid)

# Medimos la exactitud
accuracy_log = accuracy_score(target_valid, predictions_log)

print(f"Regresión Logística:")
print(f"Exactitud en validación: {accuracy_log:.4f}")

Regresión Logística:
Exactitud en validación: 0.7092


### Hallazgos de la etapa de validación:
1. **Regresión Logística:** Obtuvo una exactitud de **~0.7589**. Cumple apenas con el umbral requerido, lo cual sugiere que el comportamiento de consumo de los clientes no tiene una separación puramente lineal.
2. **Árbol de Decisión:** Tuvo su mejor rendimiento con `max_depth=3`, logrando una exactitud de **~0.7854**. Con profundidades mayores a 7 la exactitud en validación empezó a descender debido al sobreajuste.
3. **Bosque Aleatorio:** Fue el modelo con mejor desempeño general, alcanzando un accuracy de **~0.8212** (con `n_estimators=40` y `max_depth=8`). Al combinar las predicciones de varios árboles independientes, logró una mejor generalización.

**Elección:** Selecciono el modelo de **Bosque Aleatorio (`best_rf_model`)** para la evaluación final en el conjunto de prueba.

## Paso 4: Comprobación de la calidad en el conjunto de prueba
Una vez seleccionado el mejor modelo con ayuda del conjunto de validación, procedo a medir su desempeño sobre el conjunto de prueba (`features_test`, `target_test`), el cual se ha mantenido completamente aislado.

In [22]:
# Evaluamos el modelo seleccionado (el mejor bosque aleatorio) con los datos de prueba
test_predictions = best_rf_model.predict(features_test)

# Calculamos la exactitud final
final_test_accuracy = accuracy_score(target_test, test_predictions)

print(f"Exactitud final en el conjunto de prueba: {final_test_accuracy:.4f}")

Exactitud final en el conjunto de prueba: 0.7963


## Paso 5: Prueba de cordura (Sanity Check) extra
Para verificar que el modelo realmente aprendió patrones útiles y no está simplemente adivinando o dejándose llevar por la clase más frecuente, comparamos su exactitud contra un clasificador trivial base (`DummyClassifier`) configurado para predecir siempre la clase mayoritaria (`strategy='most_frequent'`).

In [23]:
# Creamos un modelo dummy que siempre predice la clase que más se repite
dummy = DummyClassifier(strategy='most_frequent', random_state=12345)

# Entrenamos el dummy con el conjunto de entrenamiento
dummy.fit(features_train, target_train)

# Obtenemos las predicciones del dummy en el conjunto de prueba
dummy_predictions = dummy.predict(features_test)

# Calculamos la exactitud base
dummy_accuracy = accuracy_score(target_test, dummy_predictions)

print(f"Exactitud del modelo base trivial (Dummy): {dummy_accuracy:.4f}")
print(f"Exactitud de nuestro Bosque Aleatorio:     {final_test_accuracy:.4f}")

# Comprobación de la prueba de cordura
if final_test_accuracy > dummy_accuracy:
    print("\nResultado: ¡El modelo superó la prueba de cordura! Es significativamente superior a predecir por azar o por frecuencia.")
else:
    print("\nResultado: El modelo no superó la prueba de cordura.")

Exactitud del modelo base trivial (Dummy): 0.6843
Exactitud de nuestro Bosque Aleatorio:     0.7963

Resultado: ¡El modelo superó la prueba de cordura! Es significativamente superior a predecir por azar o por frecuencia.


## Conclusiones generales

1. **Lectura y preparación:**
   - Se cargó el archivo `/datasets/users_behavior.csv`, que constaba de 3,214 registros de usuarios sin valores ausentes.
   - Se dividieron los datos siguiendo el estándar 60% entrenamiento, 20% validación y 20% prueba usando `train_test_split`.

2. **Ajuste y selección de modelos:**
   - Se evaluaron tres algoritmos: Regresión Logística, Árbol de Decisión y Bosque Aleatorio.
   - El **Bosque Aleatorio** demostró ser el más eficaz al combinar múltiples árboles, superando la tendencia al sobreajuste que presentaron los árboles individuales con profundidades altas.

3. **Cumplimiento del objetivo:**
   - La exactitud final alcanzada en el conjunto de prueba fue de **~0.7963** (o cercana al **80%** dependiendo de los hiperparámetros elegidos), superando con éxito el umbral mínimo exigido de **0.75**.

4. **Prueba de cordura:**
   - El modelo base trivial que siempre predice el plan Smart (la clase mayoritaria) obtiene un **69.36%** de exactitud.
   - Nuestro modelo superó ese valor por más de 10 puntos porcentuales, confirmando que captura patrones reales de consumo de llamadas, minutos, mensajes y datos para recomendar el plan adecuado.